# Imports and Setup

In [6]:
import pandas as pd
import numpy as np
import re
import yfinance as yf

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from scipy import stats

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("deep")

from langdetect import detect

print("Όλες οι βιβλιοθήκες φορτώθηκαν επιτυχώς!")

Όλες οι βιβλιοθήκες φορτώθηκαν επιτυχώς!


In [ ]:
# ============================================================
# CELL 2: Φόρτωση & Πρώτη Εξερεύνηση Dataset
# ============================================================
# Στόχος: Να κατανοήσουμε τη δομή του dataset πριν κάνουμε
# οποιαδήποτε επεξεργασία. Ελέγχουμε μέγεθος, στήλες,
# χρονικό εύρος, missing values και κατανομή ομιλιών
# ανά κεντρική τράπεζα.
# ============================================================

# Φόρτωση του dataset
df = pd.read_csv("../data/CBS_dataset_v1.0.csv")

# ── Βασικές πληροφορίες ──────────────────────────────────────
print("=" * 55)
print("  ΒΑΣΙΚΕΣ ΠΛΗΡΟΦΟΡΙΕΣ DATASET")
print("=" * 55)
print(f"  Συνολικές ομιλίες     : {len(df):,}")
print(f"  Αριθμός στηλών        : {len(df.columns)}")
print(f"  Χρονικό εύρος         : {df['Date'].min()} → {df['Date'].max()}")
print(f"  Χώρες                 : {df['Country'].nunique()}")
print(f"  Κεντρικές Τράπεζες    : {df['CentralBank'].nunique()}")

# ── Missing values ───────────────────────────────────────────
print("\n" + "=" * 55)
print("  MISSING VALUES")
print("=" * 55)
print(df.isnull().sum())

# ── Top 10 κεντρικές τράπεζες ─────────────────────────────────
# Κρίσιμο: ελέγχουμε αν η Fed έχει αρκετές ομιλίες
# για να είναι η ανάλυσή μας στατιστικά αξιόπιστη
print("\n" + "=" * 55)
print("  TOP 10 ΚΕΝΤΡΙΚΕΣ ΤΡΑΠΕΖΕΣ (αριθμός ομιλιών)")
print("=" * 55)
print(df['CentralBank'].value_counts().head(10))

# ── Κατανομή γλωσσών ─────────────────────────────────────────
print("\n" + "=" * 55)
print("  ΚΑΤΑΝΟΜΗ ΓΛΩΣΣΩΝ (Top 10)")
print("=" * 55)
print(df['Language'].value_counts().head(10))

  ΒΑΣΙΚΕΣ ΠΛΗΡΟΦΟΡΙΕΣ DATASET
  Συνολικές ομιλίες     : 35,487
  Αριθμός στηλών        : 15
  Χρονικό εύρος         : 1986-01-06 → 2023-12-29
  Χώρες                 : 131
  Κεντρικές Τράπεζες    : 143

  MISSING VALUES
URL               1652
PDF              11019
Title                0
Subtitle         13851
Date                 0
Authorname           0
Role                 0
Gender               0
CentralBank          0
Country              0
text                 0
text_original    30140
Filename             0
Language             0
Source               0
dtype: int64

  TOP 10 ΚΕΝΤΡΙΚΕΣ ΤΡΑΠΕΖΕΣ (αριθμός ομιλιών)
CentralBank
European Central Bank                        2665
Board of Governors of the Federal Reserve    2658
Deutsche Bundesbank                          1418
Bank of England                              1354
Reserve Bank of India                        1071
Bank of Italy                                1035
Central Bank of the Philippines               992
Monetary Auth

# Data Loading and Exploration

In [9]:
print(df['CentralBank'].value_counts().head(10))

CentralBank
European Central Bank                        2665
Board of Governors of the Federal Reserve    2658
Deutsche Bundesbank                          1418
Bank of England                              1354
Reserve Bank of India                        1071
Bank of Italy                                1035
Central Bank of the Philippines               992
Monetary Authority of Singapore               914
Bank of Japan                                 827
Sveriges Riksbank                             763
Name: count, dtype: int64


In [3]:
df.head()

,URL,PDF,Title,Subtitle,Date,Authorname,Role,Gender,CentralBank,Country,text,text_original,Filename,Language,Source
0,https://www.cbaruba.org/readBlob.do?id=10756,NaN,President speech Managing the Economy as if th...,NaN,2021-12-08,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Managing the Economy as if the Future Really M...,NaN,abw_10756,English,CB websites
1,https://www.cbaruba.org/readBlob.do?id=7515,NaN,Speech President of the CBA 4th Annual Future ...,NaN,2019-11-01,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Safeguarding our Future: Strategies for an Aru...,NaN,abw_7515,English,CB websites
2,https://www.cbaruba.org/readBlob.do?id=7518,NaN,Speech Symposium President Semeleer CBA,NaN,2019-09-06,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,FOSTERING ECONOMIC RESILIENCE IN ARUBA; FROM R...,NaN,abw_7518,English,CB websites
3,https://www.cbaruba.org/readBlob.do?id=7548,NaN,Integrity Koninkrijk Seminar,NaN,2016-10-28,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,"Presentation by Mrs Jeanette R. Semeleer, Pres...","Voordracht door mevrouw Jeanette R. Semeleer, ...",abw_7548,Dutch,CB websites
4,https://www.cbaruba.org/readBlob.do?id=7554,NaN,Speech by the President at the BES seminar hel...,NaN,2010-03-29,Jeanette R Semeleer,Governor,Female,Central Bank of Aruba,ABW,Ongoing changes in the supervisory landscape a...,NaN,abw_7554,English,CB websites


In [11]:
# ============================================================
# CELL 3: Επιλογή Στηλών, Φιλτράρισμα & Language Validation
# ============================================================
# Στόχος: Να κρατήσουμε μόνο τις χρήσιμες στήλες και να
# διασφαλίσουμε ότι δουλεύουμε ΜΟΝΟ με αγγλικά κείμενα.
#
# Σημαντικό: Δεν αρκεί να φιλτράρουμε βάσει της στήλης
# 'Language' — παρατηρήσαμε ότι περιέχει labeling errors
# (π.χ. ομιλία labeled ως 'Dutch' ενώ το κείμενο είναι
# αγγλικό). Κάνουμε language detection απευθείας στο
# κείμενο για 100% αξιοπιστία.
# ============================================================

def detect_language(text):
    """
    Εντοπίζει τη γλώσσα ενός κειμένου χρησιμοποιώντας
    τη βιβλιοθήκη langdetect.
    
    Παράμετροι:
    -----------
    text : str
        Το κείμενο προς ανάλυση.
    
    Επιστρέφει:
    -----------
    str
        Κωδικός γλώσσας (π.χ. 'en', 'fr', 'de') ή
        'unknown' αν δεν είναι δυνατός ο εντοπισμός.
    
    Σημειώσεις:
    -----------
    - Αναλύουμε μόνο τα πρώτα 500 chars για ταχύτητα.
    - Το try/except χειρίζεται κείμενα που είναι πολύ
      κοντά ή έχουν μη αναγνωρίσιμους χαρακτήρες.
    """
    if not isinstance(text, str) or len(text.strip()) < 20:
        return 'unknown'
    try:
        return detect(text[:500])
    except:
        return 'unknown'


# ── Επιλογή χρήσιμων στηλών ──────────────────────────────────
# Κρατάμε μόνο ό,τι χρειαζόμαστε για την ανάλυση:
# Date, text, CentralBank → βασικές για το pipeline
# Country, Authorname     → για sanity check (π.χ. Powell)
# Language, Role          → για φιλτράρισμα και τυχόν ανάλυση
cols = ['Date', 'text', 'CentralBank', 'Country', 'Authorname', 'Language', 'Role']
df = df[cols]

# ── Βήμα 1: Φίλτρο βάσει declared language ───────────────────
# Πρώτο γρήγορο φίλτρο — αφαιρεί τις προφανώς μη-αγγλικές
df_en = df[df['Language'] == 'English'].copy()

# Μετατροπή Date σε datetime — απαραίτητο για χρονοσειρά
df_en['Date'] = pd.to_datetime(df_en['Date'])

# Ταξινόμηση χρονολογικά
df_en = df_en.sort_values('Date').reset_index(drop=True)

# Αφαίρεση γραμμών χωρίς κείμενο
df_en = df_en.dropna(subset=['text'])

# ── Βήμα 2: Language Detection Validation ────────────────────
# Δεύτερο φίλτρο — επαληθεύουμε τη γλώσσα απευθείας
# στο κείμενο για να πιάσουμε τα labeling errors
print("Language detection... (2-3 λεπτά)")
df_en['detected_lang'] = df_en['text'].apply(detect_language)

# ── Αποτελέσματα σύγκρισης ───────────────────────────────────
print("\n" + "=" * 55)
print("  DECLARED vs DETECTED LANGUAGE")
print("=" * 55)
confirmed     = len(df_en[df_en['detected_lang'] == 'en'])
mismatched    = len(df_en[df_en['detected_lang'] != 'en'])
print(f"  Declared English, Detected English : {confirmed:,}")
print(f"  Declared English, Detected OTHER  : {mismatched:,}")
print(f"\n  Top detected non-English:")
print(df_en[df_en['detected_lang'] != 'en']['detected_lang'].value_counts().head(10))

# ── Βήμα 3: Drop non-English ─────────────────────────────────
df_en = df_en[df_en['detected_lang'] == 'en'].copy()

# ── Τελικό summary ────────────────────────────────────────────
print("\n" + "=" * 55)
print("  ΣΥΝΟΨΗ ΦΙΛΤΡΑΡΙΣΜΑΤΟΣ")
print("=" * 55)
print(f"  Αρχικές ομιλίες                   : {len(df):,}")
print(f"  Μετά φίλτρο 'English' label        : {confirmed + mismatched:,}")
print(f"  Μετά language detection validation : {len(df_en):,}")
print(f"  Χρονικό εύρος                      : "
      f"{df_en['Date'].min().date()} → {df_en['Date'].max().date()}")

Language detection... (2-3 λεπτά)

  DECLARED vs DETECTED LANGUAGE
  Declared English, Detected English : 30,056
  Declared English, Detected OTHER  : 84

  Top detected non-English:
detected_lang
fr    32
de    21
id    17
it     3
nl     3
tl     2
lt     2
pt     2
es     1
ca     1
Name: count, dtype: int64

  ΣΥΝΟΨΗ ΦΙΛΤΡΑΡΙΣΜΑΤΟΣ
  Αρχικές ομιλίες                   : 35,487
  Μετά φίλτρο 'English' label        : 30,140
  Μετά language detection validation : 30,056
  Χρονικό εύρος                      : 1986-01-06 → 2023-12-28


# Preprocessing: Column Selection, Filtering, and Language Validation

## Overview
This section performs critical preprocessing steps to prepare the dataset for NLP analysis. It focuses on data quality, language consistency, and temporal ordering to ensure reliable downstream processing.

## Technical Details

### 1. Column Selection
- **Purpose**: Retains only essential columns to reduce memory usage and focus analysis.
- **Selected Columns**:
  - `Date`: For temporal analysis and time-series modeling
  - `text`: Raw speech content for NLP processing
  - `CentralBank`: Institution identifier for grouping and filtering
  - `Country`, `Authorname`: Metadata for validation and sanity checks
  - `Language`, `Role`: For filtering and potential subgroup analysis
- **Implementation**: Uses pandas column indexing `df[cols]` for efficient subsetting.

### 2. Language-Based Filtering (Two-Stage Process)
- **Stage 1: Declared Language Filter**
  - Filters rows where `Language == 'English'` to remove obviously non-English content
  - **Rationale**: Quick initial filter to reduce dataset size before expensive language detection
  - **Technical Note**: Uses pandas boolean indexing `df[df['Language'] == 'English']`

- **Stage 2: Automated Language Detection Validation**
  - Applies `langdetect.detect()` to first 500 characters of each text
  - **Why 500 chars?**: Balances accuracy with computational efficiency
  - **Error Handling**: Returns 'unknown' for texts < 20 chars or detection failures
  - **Technical Implementation**: Uses `df.apply()` with custom `detect_language()` function
  - **Performance**: May take 2-3 minutes for large datasets due to per-row processing

### 3. Data Type Conversion and Sorting
- **Date Conversion**: `pd.to_datetime(df_en['Date'])` ensures proper datetime handling
- **Temporal Sorting**: `sort_values('Date')` orders data chronologically for time-series analysis
- **Index Reset**: `reset_index(drop=True)` maintains clean integer indexing

### 4. Data Cleaning
- **Missing Text Removal**: `dropna(subset=['text'])` eliminates rows without content
- **Final Language Filter**: Retains only rows where `detected_lang == 'en'`

### 5. Quality Assurance Output
- **Comparison Metrics**: Reports agreement between declared and detected languages
- **Non-English Distribution**: Shows top detected non-English languages for error analysis
- **Summary Statistics**: Provides before/after counts and date ranges

## Expected Outcomes
- Dataset reduced to high-confidence English speeches
- Chronologically ordered data ready for temporal analysis
- Comprehensive logging of filtering decisions for reproducibility

In [12]:
# Πόσες ομιλίες έχει η Fed μετά το φιλτράρισμα
fed_count = df_en[df_en['CentralBank'].str.contains(
    'Federal Reserve', case=False, na=False)]
print(f"Fed ομιλίες μετά φιλτράρισμα: {len(fed_count):,}")
print(f"Χρονικό εύρος Fed: {fed_count['Date'].min().date()} → {fed_count['Date'].max().date()}")
print(f"\nΜοναδικές ονομασίες Fed στο dataset:")
print(fed_count['CentralBank'].value_counts())

Fed ομιλίες μετά φιλτράρισμα: 6,606
Χρονικό εύρος Fed: 1986-01-06 → 2023-12-11

Μοναδικές ονομασίες Fed στο dataset:
CentralBank
Board of Governors of the Federal Reserve    2658
Federal Reserve Bank of New York              614
Federal Reserve Bank of Atlanta               527
Federal Reserve Bank of San Francisco         449
Federal Reserve Bank of Chicago               418
Federal Reserve Bank of Cleveland             353
Federal Reserve Bank of Philadelphia          282
Federal Reserve Bank of Richmond              266
Federal Reserve Bank of St Louis              252
Federal Reserve Bank of Boston                231
Federal Reserve Bank of Dallas                229
Federal Reserve Bank of Kansas City           170
Federal Reserve Bank of Minneapolis           157
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 4: Φιλτράρισμα Fed & Καθάρισμα Κειμένου
# ============================================================
# Στόχος: Να απομονώσουμε τις ομιλίες της Federal Reserve
# (Board of Governors + NY Fed) και να τις μετατρέψουμε
# σε καθαρές λίστες tokens έτοιμες για ανάλυση.
#
# Επιλογή Fed: Board of Governors + Federal Reserve Bank
# of New York — οι μόνες δύο οντότητες που συμμετέχουν
# ΠΑΝΤΑ στις ψηφοφορίες FOMC και έχουν άμεση επιρροή
# στη νομισματική πολιτική.
#
# Επιστρέφουμε ΛΙΣΤΑ tokens (όχι string) γιατί στο
# επόμενο cell χρειαζόμαστε πρόσβαση σε συγκεκριμένες
# θέσεις για το negation handling.
# ============================================================

# ── Φιλτράρισμα Fed ──────────────────────────────────────────
# Κρατάμε μόνο Board of Governors και NY Fed
fed_banks = [
    'Board of Governors of the Federal Reserve',
    'Federal Reserve Bank of New York'
]

df_fed = df_en[df_en['CentralBank'].isin(fed_banks)].copy()
df_fed = df_fed.sort_values('Date').reset_index(drop=True)

print("=" * 55)
print("  FEDERAL RESERVE — ΕΠΙΛΕΓΜΕΝΕΣ ΟΝΤΟΤΗΤΕΣ")
print("=" * 55)
print(f"  Board of Governors : "
      f"{len(df_fed[df_fed['CentralBank'] == fed_banks[0]]):,}")
print(f"  NY Fed             : "
      f"{len(df_fed[df_fed['CentralBank'] == fed_banks[1]]):,}")
print(f"  Σύνολο             : {len(df_fed):,}")
print(f"  Χρονικό εύρος      : "
      f"{df_fed['Date'].min().date()} → {df_fed['Date'].max().date()}")

# ── Καθάρισμα κειμένου ───────────────────────────────────────
stop_words = set(stopwords.words('english'))

def clean_and_tokenize(text):
    """
    Καθαρίζει κείμενο και επιστρέφει λίστα tokens
    χωρίς stopwords.
    
    Παράμετροι:
    -----------
    text : str
        Raw κείμενο ομιλίας.
    
    Επιστρέφει:
    -----------
    list
        Λίστα καθαρών tokens χωρίς stopwords.
        Κενή λίστα αν το input δεν είναι string.
    
    Βήματα:
    -------
    1. Πεζά            → "Inflation" == "inflation"
    2. Αφαίρεση αριθμών → δεν έχουν σημασιολογική αξία
    3. Αφαίρεση στίξης  → κρατάμε μόνο λέξεις
    4. Tokenization     → split σε λίστα
    5. Αφαίρεση stopwords → αφαιρούμε "the", "is", "a"...
    
    Επιστρέφουμε λίστα (όχι string) για να μπορούμε
    να ελέγχουμε συγκεκριμένες θέσεις στο negation
    handling του επόμενου cell.
    """
    if not isinstance(text, str):
        return []
    
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    
    return tokens

# ── Εφαρμογή ─────────────────────────────────────────────────
print("\nΚαθάρισμα κειμένων... (30-60 δευτερόλεπτα)")
df_fed['tokens']     = df_fed['text'].apply(clean_and_tokenize)
df_fed['word_count'] = df_fed['tokens'].apply(len)

# ── Στατιστικά ───────────────────────────────────────────────
print("\n" + "=" * 55)
print("  ΣΤΑΤΙΣΤΙΚΑ ΚΕΙΜΕΝΩΝ (μετά καθάρισμα)")
print("=" * 55)
print(f"  Μέσος αριθμός λέξεων  : {df_fed['word_count'].mean():.0f}")
print(f"  Διάμεσος λέξεων       : {df_fed['word_count'].median():.0f}")
print(f"  Ελάχιστο              : {df_fed['word_count'].min()}")
print(f"  Μέγιστο               : {df_fed['word_count'].max():,}")

# ── Φίλτρο ελάχιστου μήκους ─────────────────────────────────
# Αφαιρούμε ομιλίες < 50 λέξεων — πολύ κοντά κείμενα
# (π.χ. press releases) δεν αντιπροσωπεύουν πραγματική
# νομισματική επικοινωνία
before = len(df_fed)
df_fed = df_fed[df_fed['word_count'] >= 50].reset_index(drop=True)
after  = len(df_fed)

print("\n" + "=" * 55)
print("  ΦΙΛΤΡΟ ΕΛΑΧΙΣΤΟΥ ΜΗΚΟΥΣ (>= 50 λέξεις)")
print("=" * 55)
print(f"  Αφαιρέθηκαν (< 50 λέξεις) : {before - after:,}")
print(f"  Τελικές ομιλίες Fed        : {after:,}")

  FEDERAL RESERVE — ΕΠΙΛΕΓΜΕΝΕΣ ΟΝΤΟΤΗΤΕΣ
  🏦 Board of Governors : 2,658
  🏦 NY Fed             : 614
  📊 Σύνολο             : 3,272
  📅 Χρονικό εύρος      : 1986-01-28 → 2023-12-08

⏳ Καθάρισμα κειμένων... (30-60 δευτερόλεπτα)

  ΣΤΑΤΙΣΤΙΚΑ ΚΕΙΜΕΝΩΝ (μετά καθάρισμα)
  📝 Μέσος αριθμός λέξεων  : 1824
  📝 Διάμεσος λέξεων       : 1714
  📝 Ελάχιστο              : 65
  📝 Μέγιστο               : 38,103

  ΦΙΛΤΡΟ ΕΛΑΧΙΣΤΟΥ ΜΗΚΟΥΣ (>= 50 λέξεις)
  🗑️  Αφαιρέθηκαν (< 50 λέξεις) : 0
  ✅ Τελικές ομιλίες Fed        : 3,272


In [14]:
# Ποια είναι η μεγαλύτερη ομιλία;
max_speech = df_fed.loc[df_fed['word_count'].idxmax()]
print(f"Ημερομηνία  : {max_speech['Date'].date()}")
print(f"Συγγραφέας  : {max_speech['Authorname']}")
print(f"CentralBank : {max_speech['CentralBank']}")
print(f"Word count  : {max_speech['word_count']:,}")
print(f"\nΑρχή κειμένου:\n{max_speech['text'][:500]}")

Ημερομηνία  : 1986-06-11
Συγγραφέας  : Paul A Volcker
CentralBank : Board of Governors of the Federal Reserve
Word count  : 38,103

Αρχή κειμένου:
of the I appreciate the effort of this Subcommittee to undertake a full review of the basic approach toward banking, and bank holding company legislation and regulation. This is a large subject, filled with controversy in its particulars and with new questions arising about the philosophical underpinnings. In addition to the Bank Holding Company Act itself, the issues are relevant to the Savings and Loan Holding Company and the Glass-Steagall Acts. This statement, supplemented with detailed appe


In [15]:
# ── Κατανομή μεγέθους ομιλιών ────────────────────────────────
print("=" * 55)
print("  ΚΑΤΑΝΟΜΗ ΜΕΓΕΘΟΥΣ ΟΜΙΛΙΩΝ")
print("=" * 55)
print(f"  < 500 λέξεις   : {len(df_fed[df_fed['word_count'] < 500]):,}")
print(f"  500-1000       : {len(df_fed[(df_fed['word_count'] >= 500) & (df_fed['word_count'] < 1000)]):,}")
print(f"  1000-2000      : {len(df_fed[(df_fed['word_count'] >= 1000) & (df_fed['word_count'] < 2000)]):,}")
print(f"  2000-5000      : {len(df_fed[(df_fed['word_count'] >= 2000) & (df_fed['word_count'] < 5000)]):,}")
print(f"  5000-10000     : {len(df_fed[(df_fed['word_count'] >= 5000) & (df_fed['word_count'] < 10000)]):,}")
print(f"  > 10000        : {len(df_fed[df_fed['word_count'] > 10000]):,}")

# Πόσα outliers υπάρχουν πάνω από 10.000;
outliers = df_fed[df_fed['word_count'] > 10000]
print(f"\n  Outliers > 10.000 λέξεις:")
print(outliers[['Date', 'Authorname', 'word_count']].to_string(index=False))

  ΚΑΤΑΝΟΜΗ ΜΕΓΕΘΟΥΣ ΟΜΙΛΙΩΝ
  < 500 λέξεις   : 136
  500-1000       : 439
  1000-2000      : 1,528
  2000-5000      : 1,134
  5000-10000     : 33
  > 10000        : 2

  Outliers > 10.000 λέξεις:
      Date     Authorname  word_count
1986-06-11 Paul A Volcker       38103
1991-04-23 Alan Greenspan       13871


In [16]:
df_fed.head()

,Date,text,CentralBank,Country,Authorname,Language,Role,detected_lang,tokens,word_count
0,1986-01-28,REE (7: DOE RARY Fec 2 Bee Ratelige on Deliver...,Board of Governors of the Federal Reserve,USA,Emmett John Rice,English,Board member,en,"[ree, doe, rary, fec, bee, ratelige, delivery,...",1331
1,1986-01-29,"For release on delivery January 29, 1986 9:30 ...",Board of Governors of the Federal Reserve,USA,Paul A Volcker,English,Governor,en,"[release, delivery, january, es, feb, statemen...",2513
2,1986-02-05,For Release on Delivery 1:30 P.M. Zurich Time ...,Board of Governors of the Federal Reserve,USA,Preston Martin,English,Deputy Governor,en,"[release, delivery, pm, zurich, time, est, feb...",2296
3,1986-02-19,"For release on delivery February 19, 1986 10:0...",Board of Governors of the Federal Reserve,USA,Paul A Volcker,English,Governor,en,"[release, delivery, february, est, ea, uml, sr...",2972
4,1986-02-26,elivery AF Statement by Paul A. Volcker Chairm...,Board of Governors of the Federal Reserve,USA,Paul A Volcker,English,Governor,en,"[elivery, af, statement, paul, volcker, chairm...",815


In [18]:
for col in df_fed.columns:
    if col == 'tokens':
        # Η στήλη tokens περιέχει λίστες (unhashable) — δεν μπορούμε να κάνουμε nunique()
        print(f"\n{'='*50}")
        print(f"Column: tokens  (περιέχει λίστες — δείγμα πρώτης εγγραφής):")
        print(df_fed['tokens'].iloc[0][:10], "...")
        continue

    unique_vals = df_fed[col].nunique()
    print(f"\n{'='*50}")
    print(f"Column: {col}  ({unique_vals} unique values)")
    if unique_vals <= 30:
        print(df_fed[col].unique())
    else:
        print(f"  (πολλές τιμές — {unique_vals} συνολικά)")


Column: Date  (2616 unique values)
  (πολλές τιμές — 2616 συνολικά)

Column: text  (3272 unique values)
  (πολλές τιμές — 3272 συνολικά)

Column: CentralBank  (2 unique values)
<StringArray>
['Board of Governors of the Federal Reserve', 'Federal Reserve Bank of New York']
Length: 2, dtype: str

Column: Country  (1 unique values)
<StringArray>
['USA']
Length: 1, dtype: str

Column: Authorname  (92 unique values)
  (πολλές τιμές — 92 συνολικά)

Column: Language  (1 unique values)
<StringArray>
['English']
Length: 1, dtype: str

Column: Role  (4 unique values)
<StringArray>
['Board member', 'Governor', 'Deputy Governor', 'Senior management']
Length: 4, dtype: str

Column: detected_lang  (1 unique values)
<StringArray>
['en']
Length: 1, dtype: str

Column: tokens  (περιέχει λίστες — δείγμα πρώτης εγγραφής):
['ree', 'doe', 'rary', 'fec', 'bee', 'ratelige', 'delivery', 'ci', 'si', 'le'] ...

Column: word_count  (2007 unique values)
  (πολλές τιμές — 2007 συνολικά)


In [28]:
# ============================================================
# CELL 5b: Φόρτωση LM Dictionary & Hybrid Dictionaries
# ============================================================
# Χρησιμοποιούμε 6 κατηγορίες του LM Dictionary:
#
# → HAWKISH candidates:
#    Positive     : θετικές λέξεις για οικονομία
#    Strong_Modal : δυνατή βεβαιότητα (will, must)
#    Constraining : περιοριστικές λέξεις
#
# → DOVISH candidates:
#    Negative     : αρνητικές λέξεις για οικονομία
#    Uncertainty  : αβεβαιότητα (might, unclear)
#    Weak_Modal   : αδύναμη βεβαιότητα (may, could)
# ============================================================

# ── Φόρτωση LM ───────────────────────────────────────────────
lm = pd.read_csv("../data/LM_MasterDictionary.csv")
lm['Word'] = lm['Word'].str.lower().str.strip()

# ── Εξαγωγή κατηγοριών ───────────────────────────────────────
lm_negative     = set(lm[lm['Negative']     != 0]['Word'])
lm_positive     = set(lm[lm['Positive']     != 0]['Word'])
lm_uncertainty  = set(lm[lm['Uncertainty']  != 0]['Word'])
lm_strong_modal = set(lm[lm['Strong_Modal'] != 0]['Word'])
lm_weak_modal   = set(lm[lm['Weak_Modal']   != 0]['Word'])
lm_constraining = set(lm[lm['Constraining'] != 0]['Word'])

print("=" * 55)
print("  LM DICTIONARY — ΚΑΤΗΓΟΡΙΕΣ")
print("=" * 55)
print(f"  Negative     : {len(lm_negative):,}")
print(f"  Positive     : {len(lm_positive):,}")
print(f"  Uncertainty  : {len(lm_uncertainty):,}")
print(f"  Strong_Modal : {len(lm_strong_modal):,}")
print(f"  Weak_Modal   : {len(lm_weak_modal):,}")
print(f"  Constraining : {len(lm_constraining):,}")

# ── Domain-specific λέξεις ────────────────────────────────────
hawkish_domain = {
    # Πληθωρισμός
    'inflation', 'inflationary', 'overheating',
    'overheat', 'overheated', 'price pressures',
    'above target', 'supply constraints',
    # Σύσφιξη
    'tighten', 'tightening', 'tightened', 'restrictive',
    'normalize', 'normalizing', 'normalization',
    'front-loading', 'front-load',
    # Επιτόκια
    'hike', 'hikes', 'rate hike', 'rate hikes',
    'raise rates', 'raising rates', 'rate increase',
    'three-quarter',
    # Tapering
    'taper', 'tapering', 'tapered',
    # Στάση
    'hawkish', 'vigilant', 'vigilance',
    'aggressive', 'aggressively', 'forceful',
    'tight labor',
    # Άλλα
    'combat inflation'
}

dovish_domain = {
    # Χαλάρωση
    'accommodative', 'accommodation', 'easing',
    'stimulus', 'stimulative', 'expansionary',
    'soft landing',
    # Μειώσεις
    'rate cut', 'rate cuts', 'cutting rates',
    'lower rates', 'lowering rates', 'hold rates',
    # Αγορά εργασίας
    'unemployment', 'slack', 'underemployment',
    'maximum employment', 'below potential', 'output gap',
    # Ύφεση
    'recession', 'slowdown', 'downside risks',
    'weakening', 'deterioration', 'headwinds',
    # QE
    'quantitative easing', 'asset purchases',
    'bond buying', 'purchase program',
    # Στάση
    'dovish', 'patient', 'patience',
    'gradual', 'gradually', 'supportive',
    'below target', 'forward guidance'
}
# ── Φιλτράρισμα LM words ─────────────────────────────────────

# Λέξεις που το LM βάζει ως Negative αλλά για εμάς
# είναι hawkish — τις εξαιρούμε από το dovish
lm_negative_exclude = {
    'inflation', 'inflationary', 'overheating',
    'excessive', 'aggressive', 'risk', 'risks',
    'tighten', 'tightening', 'restrictive'
}

# Οικονομικά relevant negative words → dovish
economic_negative = {
    'weak', 'weakness', 'weaken', 'weakening',
    'decline', 'declining', 'deteriorate', 'deteriorating',
    'worsen', 'worsening', 'depressed',
    'adverse', 'adversely', 'vulnerable', 'vulnerability',
    'contraction', 'contracting',
    'sluggish', 'stagnant', 'stagnation',
    'distress', 'distressed', 'hardship',
    'downturn', 'downward', 'downside'
}

# Uncertainty words → dovish
uncertainty_keep = {
    'uncertain', 'uncertainty', 'unclear',
    'unpredictable', 'volatile', 'volatility',
    'instability', 'unstable', 'ambiguous'
}

# Strong Modal → hawkish (δυνατή βεβαιότητα)
# Φιλτράρουμε μόνο αυτές που έχουν νόημα στο context
strong_modal_keep = {
    'require', 'required', 'necessary', 'necessitate'
}

# Weak Modal → dovish (αδύναμη βεβαιότητα)
weak_modal_keep = {
    'likely', 'unlikely'
}

# ── Τελικά Hybrid Dictionaries ────────────────────────────────
final_hawkish = (
    hawkish_domain                              # Domain-specific
    | (lm_positive & hawkish_domain)            # LM Positive confirmed
    | (lm_strong_modal & strong_modal_keep)     # LM Strong Modal
    | (lm_constraining & hawkish_domain)        # LM Constraining
)

final_dovish = (
    dovish_domain                               # Domain-specific
    | (lm_negative & economic_negative)         # LM Negative filtered
    | (lm_uncertainty & uncertainty_keep)       # LM Uncertainty filtered
    | (lm_weak_modal & weak_modal_keep)         # LM Weak Modal
)

# ── Overlap Check & Resolution ────────────────────────────────
overlap = final_hawkish & final_dovish
if overlap:
    print(f"\nOverlap εντοπίστηκε — αφαιρείται από dovish:")
    print(overlap)
    final_dovish = final_dovish - overlap
else:
    print(f"\nΚανένα overlap")

# ── Αποτελέσματα ──────────────────────────────────────────────
print("\n" + "=" * 55)
print("  ΤΕΛΙΚΑ HYBRID DICTIONARIES")
print("=" * 55)
print(f"    Hawkish words : {len(final_hawkish)}")
print(f"     ├── Domain                   : {len(hawkish_domain)}")
print(f"     ├── LM Strong Modal          : "
      f"{len(lm_strong_modal & strong_modal_keep)}")
print(f"     └── LM Constraining          : "
      f"{len(lm_constraining & hawkish_domain)}")
print(f"\n     Dovish words  : {len(final_dovish)}")
print(f"     ├── Domain                   : {len(dovish_domain)}")
print(f"     ├── LM Negative (filtered)   : "
      f"{len(lm_negative & economic_negative)}")
print(f"     ├── LM Uncertainty (filtered): "
      f"{len(lm_uncertainty & uncertainty_keep)}")
print(f"     └── LM Weak Modal            : "
      f"{len(lm_weak_modal & weak_modal_keep)}")
print(f"\n  Τελικό overlap : {len(overlap)}")

  LM DICTIONARY — ΚΑΤΗΓΟΡΙΕΣ
  Negative     : 2,355
  Positive     : 354
  Uncertainty  : 297
  Strong_Modal : 19
  Weak_Modal   : 27
  Constraining : 184

Κανένα overlap

  ΤΕΛΙΚΑ HYBRID DICTIONARIES
    Hawkish words : 36
     ├── Domain                   : 36
     ├── LM Strong Modal          : 0
     └── LM Constraining          : 1

     Dovish words  : 68
     ├── Domain                   : 37
     ├── LM Negative (filtered)   : 24
     ├── LM Uncertainty (filtered): 8
     └── LM Weak Modal            : 0

  Τελικό overlap : 0


In [ ]:
# ============================================================
# CELL 6: IDF-Weighted Hawk-Dove Score
# ============================================================
# Αντί για χειροκίνητα βάρη (1,2,3) χρησιμοποιούμε
# IDF weights — αυτόματα υψηλότερο βάρος σε σπάνιες
# λέξεις που εμφανίζονται μόνο σε συγκεκριμένες ομιλίες.
#
# IDF(λέξη) = log(N / df) όπου:
#   N  = συνολικές ομιλίες
#   df = ομιλίες που περιέχουν τη λέξη
#
# Αποτέλεσμα:
#   "inflation"        IDF=1.80 → χαμηλό βάρος (παντού)
#   "rate hike"        IDF=9.09 → υψηλό βάρος (σπάνια)
#   "quantitative eas" IDF=9.09 → υψηλό βάρος (σπάνια)
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

# ── Ορισμός dictionaries ──────────────────────────────────────
final_hawkish = {
    # Επιτόκια — ξεκάθαρη σύσφιξη
    'rate hike', 'rate hikes', 'rate increase',
    'raise rates', 'raising rates',
    'hike', 'hikes',
    # Σύσφιξη
    'tightening', 'tighten', 'tightened',
    'restrictive', 'restriction',
    # Normalization
    'normalize', 'normalizing', 'normalization',
    # Tapering
    'tapering', 'taper', 'tapered',
    # Επιθετική πολιτική
    'front-loading', 'front-load',
    'three-quarter', 'aggressive', 'forceful',
    # Στάση
    'hawkish', 'vigilant', 'vigilance',
    # Specific hawkish phrases
    'above target', 'tight labor',
    'combat inflation', 'supply constraints'
}

final_dovish = {
    # Επιτόκια — ξεκάθαρη μείωση
    'rate cut', 'rate cuts', 'cutting rates',
    'lower rates', 'lowering rates', 'hold rates',
    # QE — ξεκάθαρη χαλάρωση
    'quantitative easing', 'asset purchases',
    'bond buying', 'purchase program',
    # Χαλαρή στάση
    'accommodative', 'accommodation',
    'dovish', 'easing',
    # Δέσμευση για αργή κίνηση
    'patience', 'patient',
    'gradual', 'gradually',
    'forward guidance', 'soft landing',
    # Specific dovish phrases
    'below target', 'output gap',
    'below potential', 'maximum employment',
    'downside risks', 'hold rates'
}

# Overlap check
overlap = final_hawkish & final_dovish
if overlap:
    print(f"  Overlap: {overlap}")
    final_dovish = final_dovish - overlap
else:
    print(" Κανένα overlap")

# ── Υπολογισμός IDF weights ───────────────────────────────────
print("⏳ Υπολογισμός IDF weights...")

# Μετατρέπουμε tokens σε strings για TfidfVectorizer
corpus = df_fed['tokens'].apply(lambda x: ' '.join(x)).tolist()

# Vocabulary = union των δύο dictionaries
# Χρησιμοποιούμε _ αντί για κενό για multi-word terms
all_terms = list(final_hawkish | final_dovish)

# Για multi-word terms (π.χ. "rate hike") χρειαζόμαστε
# ngram_range=(1,3) ώστε να τα πιάσει ο vectorizer
vectorizer = TfidfVectorizer(
    vocabulary   = all_terms,
    ngram_range  = (1, 3),
    norm         = None,    # Δεν κάνουμε L2 normalization
    use_idf      = True,
    smooth_idf   = True
)

tfidf_matrix = vectorizer.fit_transform(corpus)

# IDF dictionary: {λέξη: idf_value}
idf_values = dict(zip(
    vectorizer.get_feature_names_out(),
    vectorizer.idf_
))

print("\n" + "=" * 55)
print("  IDF VALUES — ΕΠΙΛΕΓΜΕΝΕΣ ΛΕΞΕΙΣ")
print("=" * 55)
check_words = [
    'rate hike', 'tightening', 'three-quarter',
    'tapering', 'restrictive', 'hawkish',
    'quantitative easing', 'forward guidance',
    'accommodative', 'rate cut'
]
for word in check_words:
    if word in idf_values:
        bar = "█" * int(idf_values[word])
        print(f"  {word:<25} : {idf_values[word]:.4f}  {bar}")

# ── Negation words ────────────────────────────────────────────
negation_words = {
    'not', 'no', 'never', 'neither', 'nor',
    'cannot', "can't", "won't", "doesn't", "don't",
    "didn't", "isn't", "aren't", "wasn't", "weren't",
    'without', 'hardly', 'barely', 'scarcely'
}

NEGATION_WINDOW = 5

# ── IDF-Weighted Score Function ───────────────────────────────
def compute_hawk_dove_score(tokens):
    """
    Υπολογίζει IDF-weighted Hawk-Dove Score ομιλίας.
    
    Παράμετροι:
    -----------
    tokens : list
        Λίστα καθαρών tokens.
    
    Επιστρέφει:
    -----------
    dict:
        'score'         : float — κανονικοποιημένο score
        'hawkish_score' : float — σταθμισμένα hawkish hits
        'dovish_score'  : float — σταθμισμένα dovish hits
        'negations'     : int   — negations που εντοπίστηκαν
    
    Μεθοδολογία:
    ------------
    Για κάθε match: βάρος = IDF(λέξη)
    Score = Σ(hawkish × IDF) - Σ(dovish × IDF)
            ─────────────────────────────────────
                      total_words × 1000
    
    Αποτέλεσμα: γενικές λέξεις (χαμηλό IDF) συνεισφέρουν
    λίγο, specific πολιτικές λέξεις (υψηλό IDF) πολύ.
    """
    if not tokens or len(tokens) == 0:
        return {'score': 0, 'hawkish_score': 0,
                'dovish_score': 0, 'negations': 0}
    
    hawkish_score = 0
    dovish_score  = 0
    negations     = 0
    total_words   = len(tokens)
    
    # Ελέγχουμε single tokens και bigrams/trigrams
    for i, token in enumerate(tokens):
        
        # Single token check
        candidates = [token]
        
        # Bigram
        if i + 1 < len(tokens):
            candidates.append(f"{token} {tokens[i+1]}")
        
        # Trigram
        if i + 2 < len(tokens):
            candidates.append(f"{token} {tokens[i+1]} {tokens[i+2]}")
        
        for candidate in candidates:
            is_hawkish = candidate in final_hawkish
            is_dovish  = candidate in final_dovish
            
            if not is_hawkish and not is_dovish:
                continue
            
            # IDF weight
            idf_weight = idf_values.get(candidate, 1.0)
            
            # Negation check
            start        = max(0, i - NEGATION_WINDOW)
            context      = tokens[start:i]
            has_negation = any(
                neg in context for neg in negation_words
            )
            
            if has_negation:
                negations += 1
                if is_hawkish:
                    dovish_score  += idf_weight
                else:
                    hawkish_score += idf_weight
            else:
                if is_hawkish:
                    hawkish_score += idf_weight
                else:
                    dovish_score  += idf_weight
    
    score = (hawkish_score - dovish_score) / total_words * 1000
    
    return {
        'score'         : score,
        'hawkish_score' : hawkish_score,
        'dovish_score'  : dovish_score,
        'negations'     : negations
    }

# ── Εφαρμογή ─────────────────────────────────────────────────
print("\n Υπολογισμός IDF-Weighted Score... (1-2 λεπτά)")
results = df_fed['tokens'].apply(compute_hawk_dove_score)

df_fed['hawk_dove_score'] = results.apply(lambda x: x['score'])
df_fed['hawkish_score']   = results.apply(lambda x: x['hawkish_score'])
df_fed['dovish_score']    = results.apply(lambda x: x['dovish_score'])
df_fed['negations']       = results.apply(lambda x: x['negations'])

# ── Στατιστικά ───────────────────────────────────────────────
print("\n" + "=" * 55)
print("  ΣΤΑΤΙΣΤΙΚΑ IDF-WEIGHTED HAWK-DOVE SCORE")
print("=" * 55)
print(f"   Μέσος όρος   : {df_fed['hawk_dove_score'].mean():.4f}")
print(f"   Διάμεσος     : {df_fed['hawk_dove_score'].median():.4f}")
print(f"   Std deviation: {df_fed['hawk_dove_score'].std():.4f}")
print(f"   Min          : {df_fed['hawk_dove_score'].min():.4f}")
print(f"   Max          : {df_fed['hawk_dove_score'].max():.4f}")
print(f"\n  🦅 Hawkish (score > 0): "
      f"{(df_fed['hawk_dove_score'] > 0).sum():,}")
print(f"    Dovish  (score < 0): "
      f"{(df_fed['hawk_dove_score'] < 0).sum():,}")
print(f"    Ουδέτερες (score=0): "
      f"{(df_fed['hawk_dove_score'] == 0).sum():,}")
print(f"\n   Negations: {df_fed['negations'].sum():,}")

# ── Sanity Check ──────────────────────────────────────────────
print("\n" + "=" * 55)
print("  TOP 5 HAWKISH ΟΜΙΛΙΕΣ")
print("=" * 55)
print(df_fed[['Date', 'Authorname', 'hawk_dove_score']]
      .nlargest(5, 'hawk_dove_score').to_string(index=False))

print("\n" + "=" * 55)
print("  TOP 5 DOVISH ΟΜΙΛΙΕΣ")
print("=" * 55)
print(df_fed[['Date', 'Authorname', 'hawk_dove_score']]
      .nsmallest(5, 'hawk_dove_score').to_string(index=False))

# ── Σύγκριση με προηγούμενα scores ───────────────────────────
print("\n" + "=" * 55)
print("  ΣΥΓΚΡΙΣΗ SCORES — PROBLEMATIC SPEECHES")
print("=" * 55)
for name, date in [('Frederic S Mishkin',  '2007-10-20'),
                   ('Philip N Jefferson',  '2023-11-14'),
                   ('Jerome H Powell',     '2022-08-26')]:
    row = df_fed[
        (df_fed['Authorname'] == name) &
        (df_fed['Date'] == date)
    ]
    if len(row) > 0:
        score = row.iloc[0]['hawk_dove_score']
        print(f"  {name} ({date}): {score:.4f}")

✅ Κανένα overlap
⏳ Υπολογισμός IDF weights...

  IDF VALUES — ΕΠΙΛΕΓΜΕΝΕΣ ΛΕΞΕΙΣ
  rate hike                 : 6.5285  ██████
  tightening                : 2.6258  ██
  three-quarter             : 9.0935  █████████
  tapering                  : 5.3799  █████
  restrictive               : 3.6087  ███
  hawkish                   : 7.3017  ███████
  quantitative easing       : 4.5826  ████
  forward guidance          : 3.9518  ███
  accommodative             : 3.0628  ███
  rate cut                  : 6.0977  ██████

⏳ Υπολογισμός IDF-Weighted Score... (1-2 λεπτά)

  ΣΤΑΤΙΣΤΙΚΑ IDF-WEIGHTED HAWK-DOVE SCORE
  📊 Μέσος όρος   : -1.0300
  📊 Διάμεσος     : 0.0000
  📊 Std deviation: 9.4194
  📊 Min          : -127.7969
  📊 Max          : 92.1165

  🦅 Hawkish (score > 0): 975
  🕊️  Dovish  (score < 0): 1,121
  ⚖️  Ουδέτερες (score=0): 1,176

  🔄 Negations: 67

  TOP 5 HAWKISH ΟΜΙΛΙΕΣ
      Date           Authorname  hawk_dove_score
2023-05-24 Christopher J Waller        92.116541
2022-06-18 Chris

In [41]:
# Διερεύνηση ομιλιών με score=0
print("=" * 55)
print("  ΔΙΕΡΕΥΝΗΣΗ ΟΜΙΛΙΩΝ ΜΕ SCORE=0")
print("=" * 55)

zero_speeches = df_fed[df_fed['hawk_dove_score'] == 0].copy()

# Ελέγχουμε αν τα tokens τους περιέχουν ΚΑΜΙΑ dictionary λέξη
def count_dict_hits(tokens):
    hits = 0
    for i, token in enumerate(tokens):
        candidates = [token]
        if i + 1 < len(tokens):
            candidates.append(f"{token} {tokens[i+1]}")
        if i + 2 < len(tokens):
            candidates.append(f"{token} {tokens[i+1]} {tokens[i+2]}")
        for c in candidates:
            if c in final_hawkish or c in final_dovish:
                hits += 1
    return hits

print("⏳ Υπολογισμός hits για score=0 ομιλίες...")
zero_speeches['total_hits'] = zero_speeches['tokens'].apply(
    count_dict_hits
)

print(f"\n  Ομιλίες με 0 dictionary hits  : "
      f"{(zero_speeches['total_hits'] == 0).sum():,}")
print(f"  Ομιλίες με hits αλλά score=0  : "
      f"{(zero_speeches['total_hits'] > 0).sum():,}")

# Κατανομή ανά δεκαετία
print(f"\n  Κατανομή ανά δεκαετία:")
zero_speeches['decade'] = (
    zero_speeches['Date'].dt.year // 10 * 10
)
print(zero_speeches['decade'].value_counts().sort_index())

# Κατανομή ανά CentralBank
print(f"\n  Κατανομή ανά CentralBank:")
print(zero_speeches['CentralBank'].value_counts())

  ΔΙΕΡΕΥΝΗΣΗ ΟΜΙΛΙΩΝ ΜΕ SCORE=0
⏳ Υπολογισμός hits για score=0 ομιλίες...

  Ομιλίες με 0 dictionary hits  : 1,175
  Ομιλίες με hits αλλά score=0  : 1

  Κατανομή ανά δεκαετία:
decade
1980     97
1990    301
2000    333
2010    318
2020    127
Name: count, dtype: int64

  Κατανομή ανά CentralBank:
CentralBank
Board of Governors of the Federal Reserve    938
Federal Reserve Bank of New York             238
Name: count, dtype: int64


In [42]:
# ── Relevance Filter ──────────────────────────────────────────
# Κρατάμε μόνο ομιλίες με τουλάχιστον 1 dictionary hit
# Αυτές είναι οι ομιλίες που περιέχουν νομισματικό σήμα

before = len(df_fed)
df_fed_filtered = df_fed[df_fed['hawk_dove_score'] != 0].copy()
after  = len(df_fed_filtered)

print("=" * 55)
print("  RELEVANCE FILTERING")
print("=" * 55)
print(f"  Πριν  : {before:,} ομιλίες")
print(f"  Μετά  : {after:,} ομιλίες")
print(f"  Αφαιρέθηκαν : {before - after:,} ομιλίες (score=0)")
print(f"\n  Νέα κατανομή:")
print(f"  🦅 Hawkish (score > 0): "
      f"{(df_fed_filtered['hawk_dove_score'] > 0).sum():,}")
print(f"  🕊️  Dovish  (score < 0): "
      f"{(df_fed_filtered['hawk_dove_score'] < 0).sum():,}")
print(f"\n  Νέος μέσος όρος : "
      f"{df_fed_filtered['hawk_dove_score'].mean():.4f}")
print(f"  Νέος διάμεσος   : "
      f"{df_fed_filtered['hawk_dove_score'].median():.4f}")

# Κατανομή ανά δεκαετία μετά το φιλτράρισμα
print(f"\n  Ομιλίες ανά δεκαετία (μετά):")
df_fed_filtered['decade'] = (
    df_fed_filtered['Date'].dt.year // 10 * 10
)
print(df_fed_filtered['decade'].value_counts().sort_index())

  RELEVANCE FILTERING
  Πριν  : 3,272 ομιλίες
  Μετά  : 2,096 ομιλίες
  Αφαιρέθηκαν : 1,176 ομιλίες (score=0)

  Νέα κατανομή:
  🦅 Hawkish (score > 0): 975
  🕊️  Dovish  (score < 0): 1,121

  Νέος μέσος όρος : -1.6079
  Νέος διάμεσος   : -0.7615

  Ομιλίες ανά δεκαετία (μετά):
decade
1980    121
1990    462
2000    656
2010    601
2020    256
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 7: Μηνιαία Χρονοσειρά & ADF Stationarity Test
# ============================================================
# Στόχος: Να μετατρέψουμε τα individual speech scores σε
# μηνιαία χρονοσειρά έτοιμη για στατιστική ανάλυση.
#
# Βήματα:
# ───────
# 1. Minimum threshold: αποκλείουμε μήνες < 2 ομιλίες
# 2. Μηνιαίος μέσος όρος score
# 3. ADF Test για stationarity
# 4. First-differencing αν non-stationary
# ============================================================

# ── Βήμα 1: Μηνιαία Συγκέντρωση ──────────────────────────────
df_fed_filtered = df_fed_filtered.set_index('Date')

# Μέσος όρος score ανά μήνα
monthly_score = (df_fed_filtered['hawk_dove_score']
                 .resample('ME')
                 .mean())

# Αριθμός ομιλιών ανά μήνα
monthly_count = (df_fed_filtered['hawk_dove_score']
                 .resample('ME')
                 .count())

# ── Βήμα 2: Minimum Threshold ─────────────────────────────────
# Αποκλείουμε μήνες με λιγότερες από 2 ομιλίες
# Ένας μήνας με 1 ομιλία δεν αντιπροσωπεύει αξιόπιστα
# τη συνολική στάση της Fed
MIN_SPEECHES = 2

valid_months  = monthly_count[monthly_count >= MIN_SPEECHES].index
monthly_score = monthly_score[monthly_score.index.isin(valid_months)]
monthly_count = monthly_count[monthly_count.index.isin(valid_months)]

print("=" * 55)
print("  ΜΗΝΙΑΙΑ ΧΡΟΝΟΣΕΙΡΑ")
print("=" * 55)
print(f"   Χρονικό εύρος    : "
      f"{monthly_score.index.min().date()} → "
      f"{monthly_score.index.max().date()}")
print(f"   Συνολικοί μήνες  : {len(monthly_score)}")
print(f"   Μέσος όρος score : {monthly_score.mean():.4f}")
print(f"   Std deviation    : {monthly_score.std():.4f}")
print(f"\n  Μέσος αριθμός ομιλιών/μήνα: "
      f"{monthly_count.mean():.1f}")
print(f"  Min ομιλίες/μήνα          : "
      f"{monthly_count.min()}")
print(f"  Max ομιλίες/μήνα          : "
      f"{monthly_count.max()}")

# ── Βήμα 3: ADF Test — Stationarity Check ────────────────────
# Null hypothesis: η χρονοσειρά είναι NON-stationary
# p-value < 0.05 → απορρίπτουμε null → STATIONARY 
# p-value > 0.05 → δεν απορρίπτουμε → NON-STATIONARY 
print("\n" + "=" * 55)
print("  ADF TEST — STATIONARITY CHECK")
print("=" * 55)

adf_result = adfuller(monthly_score.dropna())

print(f"  ADF Statistic : {adf_result[0]:.4f}")
print(f"  p-value       : {adf_result[1]:.4f}")
print(f"  Critical values:")
for key, val in adf_result[4].items():
    print(f"    {key}: {val:.4f}")

if adf_result[1] < 0.05:
    print(f"\n   STATIONARY (p < 0.05)")
    print(f"  → Χρησιμοποιούμε τα επίπεδα (levels) απευθείας")
    monthly_score_final = monthly_score.copy()
    is_differenced = False
else:
    print(f"\n   NON-STATIONARY (p > 0.05)")
    print(f"  → Εφαρμόζουμε first-differencing")
    monthly_score_final = monthly_score.diff().dropna()
    is_differenced = True
    
    # Επαναλαμβάνουμε ADF στις first differences
    adf_result2 = adfuller(monthly_score_final.dropna())
    print(f"\n  ADF Test (first differences):")
    print(f"  ADF Statistic : {adf_result2[0]:.4f}")
    print(f"  p-value       : {adf_result2[1]:.4f}")
    if adf_result2[1] < 0.05:
        print(f"   STATIONARY μετά first-differencing")
    else:
        print(f"    Ακόμα non-stationary — χρειάζεται περαιτέρω ανάλυση")

print(f"\n   Τελική χρονοσειρά:")
print(f"  Μήνες        : {len(monthly_score_final)}")
print(f"  Μέσος όρος   : {monthly_score_final.mean():.4f}")
print(f"  Std deviation: {monthly_score_final.std():.4f}")

  ΜΗΝΙΑΙΑ ΧΡΟΝΟΣΕΙΡΑ
  📅 Χρονικό εύρος    : 1986-02-28 → 2023-12-31
  📊 Συνολικοί μήνες  : 398
  📊 Μέσος όρος score : -1.5159
  📊 Std deviation    : 7.5572

  Μέσος αριθμός ομιλιών/μήνα: 5.2
  Min ομιλίες/μήνα          : 2
  Max ομιλίες/μήνα          : 17

  ADF TEST — STATIONARITY CHECK
  ADF Statistic : -3.5356
  p-value       : 0.0071
  Critical values:
    1%: -3.4471
    5%: -2.8689
    10%: -2.5707

  ✅ STATIONARY (p < 0.05)
  → Χρησιμοποιούμε τα επίπεδα (levels) απευθείας

  📊 Τελική χρονοσειρά:
  Μήνες        : 398
  Μέσος όρος   : -1.5159
  Std deviation: 7.5572
